# Siglet‑Qubit Kernel POC

**Date:** 2025‑04‑21  
**Author:** Luiz Frias

---

## 📖 Summary / Mission

We’re building a **classical prototype** of the Siglet‑Qubit kernel:
1. **Define** `SigletQubit(theta, τ, ε, μ, c)`
2. **Sweep** resonance (θ) and coherence (τ) across time to map “truth‑score” decay
3. **Cluster** stable regions to identify a finite siglet alphabet
4. **Visualize** results with decay curves and heatmaps

This notebook is the **ground floor**—today’s deliverable is a self‑contained classical sim. Tomorrow we’ll port core gates into a 1‑qubit quantum toy.

---

## 🎯 Objectives

- ✅ Implement `SigletQubit` class  
- ✅ Generate decay curves for sample θ values  
- ✅ Produce a heatmap of T(s,t) over (θ,τ) grid  
- 🔲 Cluster stable siglet regions  
- 🔲 Export CSV for memo & outreach  

---

## 🛠 Tech Stack & Constraints

- **Language:** Python 3.9+  
- **Libraries:** `numpy`, `matplotlib`, (`scikit‑learn` for clustering)  
- **Environment:** Local Jupyter on M2 MacBook Air (no cloud)  
- **Style:** Clean, modular cells; extensive comments; version control via git

---

## 🔗 Next Steps

1. (Today) Complete classical sweep & visualization  
2. (Tomorrow) Add 1‑qubit quantum demo with Qiskit or PennyLane  
3. Draft 1‑page POC memo and attach notebook as appendix  
4. Begin advisor/fellowship outreach with results  

---

# PRD: Siglet‑Qubit Kernel Prototype

## Background  
RealityOS needs a **foundational symbolic unit** (“siglet”) that carries resonance, temporality, ethics, modality, and compression. We model it classically first, then port to photonic qubits.

## Functional Requirements  
1. **Data structure**: `SigletQubit(theta, tau, eps_norm, mu, c)`  
2. **Truth‑score function**: `T(s, t) = cos(theta) * tau^t * eps_norm * mu * c`  
3. **Parameter sweep** over  
   - `theta ∈ [0,π]` (50 steps)  
   - `tau ∈ [0.8,1.0]` (20 steps)  
4. **Visualization**  
   - Decay curves for selected θ  
   - 2D heatmap of T(s,t) across (θ,τ)

## Non‑Functional Requirements  
- Runs in < 30 s on local M2  
- Jupyter cells self‑documented  
- Results exportable (CSV + PNG)  

## Future Extensions  
- Quantum toy via Qiskit / PennyLane  
- Moral‑axis clustering (scikit‑learn)  
- Integration into RealityOS agent

<!--
SYSTEM: You are an expert AI research partner. The notebook’s goal is to prototype “SigletQubit” as a classical data structure, sweep parameters θ and τ to compute a truth‑score decay T(s,t), visualize results, and cluster stable regions. Use Python, numpy, matplotlib. Keep cells modular, documented, and ready for a future quantum extension. Provide code only; minimize prose.
-->

In [ ]:
# Libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import mlflow
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import time
from sklearn.metrics import confusion_matrix
from tslearn.metrics import cdist_dtw
from sklearn.cluster import DBSCAN, SpectralClustering
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# Set up MLflow experiment
mlflow.set_experiment("SigletQubit-Classical-Sim")

In [ ]:
# set working directory to the project root
ROOT_DIR = "/Users/luizfrias/CursorAI/data-science/siglet_architecture"

# Define project paths
DATA_DIR = os.path.join(ROOT_DIR, "data/processed")
FIGURES_DIR = os.path.join(ROOT_DIR, "reports/figures")
MODELS_DIR = os.path.join(ROOT_DIR, "models")
NOTES_DIR = os.path.join(ROOT_DIR, "reports/notes")

# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(NOTES_DIR, exist_ok=True)

In [ ]:
# Define the dataclass for the qubit
class SigletQubit:
    def __init__(self, theta, tau, eps_norm, mu, c):
        self.theta = theta  # resonance angle
        self.tau = tau  # coherence factor
        self.eps_norm = eps_norm  # ||ε||
        self.mu = mu  # modality fidelity
        self.c = c  # compression ratio

    def resonance(self):
        return np.cos(self.theta)

    def truth_score(self, t):
        r = self.resonance()
        return r * (self.tau**t) * self.eps_norm * self.mu * self.c

In [ ]:
# Parameter grid
thetas = np.linspace(0, np.pi, 50)
taus = np.linspace(0.8, 1.0, 20)
t_max = 10

In [ ]:
# Start MLflow run
with mlflow.start_run(run_name="decay_curves") as run:
    # Log parameters
    mlflow.log_param("theta_range", [0, np.pi])
    mlflow.log_param("tau_range", [0.8, 1.0])
    mlflow.log_param("t_max", t_max)
    mlflow.log_param("eps_norm", 1.0)
    mlflow.log_param("mu", 0.8)
    mlflow.log_param("c", 0.7)

    # Sweep & plot a few curves
    plt.figure(figsize=(8, 5))

    sample_thetas = [0, np.pi / 4, np.pi / 2]
    for theta in sample_thetas:
        sq = SigletQubit(theta=theta, tau=0.9, eps_norm=1.0, mu=0.8, c=0.7)
        scores = [sq.truth_score(t) for t in range(t_max + 1)]
        plt.plot(range(t_max + 1), scores, label=f"θ={theta:.2f}")

        # Log metrics for each theta
        mlflow.log_metric(f"initial_score_theta_{theta:.2f}", scores[0])
        mlflow.log_metric(f"final_score_theta_{theta:.2f}", scores[-1])
        mlflow.log_metric(
            f"decay_rate_theta_{theta:.2f}", (scores[0] - scores[-1]) / t_max
        )

    plt.xlabel("Time (t)")
    plt.ylabel("Truth Score T(s, t)")
    plt.title("Siglet‑Qubit Decay Curves")
    plt.legend()

    # Save figure locally
    decay_fig_path = os.path.join(FIGURES_DIR, "decay_curves.png")
    plt.savefig(decay_fig_path, dpi=300)

    # Log artifact
    mlflow.log_artifact(decay_fig_path)

    plt.show()

In [ ]:
# Generate the heatmap of T(s,t) over (θ,τ) grid
with mlflow.start_run(run_name="heatmap_generation") as run:
    start_time = time.time()

    # Log parameters
    mlflow.log_param("theta_steps", len(thetas))
    mlflow.log_param("tau_steps", len(taus))
    mlflow.log_param("t_eval", 5)  # Time point to evaluate

    t_eval = 5  # Time point to evaluate (can be adjusted)
    truth_scores = np.zeros((len(thetas), len(taus)))

    for i, theta in enumerate(thetas):
        for j, tau in enumerate(taus):
            sq = SigletQubit(theta=theta, tau=tau, eps_norm=1.0, mu=0.8, c=0.7)
            truth_scores[i, j] = sq.truth_score(t_eval)

    # Log performance metric
    computation_time = time.time() - start_time
    mlflow.log_metric("computation_time_seconds", computation_time)
    mlflow.log_metric("mean_truth_score", np.mean(truth_scores))
    mlflow.log_metric("max_truth_score", np.max(truth_scores))
    mlflow.log_metric("min_truth_score", np.min(truth_scores))

    # Create the heatmap
    plt.figure(figsize=(10, 8))
    # Store the heatmap object in a variable
    hm = sns.heatmap(
        truth_scores,
        xticklabels=np.round(taus, 2),
        yticklabels=np.round(thetas, 2),
        cmap="viridis",
        cbar_kws={"label": "Truth Score"},
    )  # Add colorbar label directly here

    plt.xlabel("Coherence (τ)")
    plt.ylabel("Resonance (θ)")
    plt.title(f"Truth Score T(s,t) at t={t_eval}")
    # Remove this line as the colorbar is already created by sns.heatmap
    # plt.colorbar(label='Truth Score')
    plt.tight_layout()

    # Save figure locally
    heatmap_fig_path = os.path.join(FIGURES_DIR, "siglet_heatmap.png")
    plt.savefig(heatmap_fig_path, dpi=300)

    # Log artifact
    mlflow.log_artifact(heatmap_fig_path)

    # Save data for later use
    truth_scores_path = os.path.join(DATA_DIR, "truth_scores.npy")
    np.save(truth_scores_path, truth_scores)
    mlflow.log_artifact(truth_scores_path)

    plt.show()

In [ ]:
# Cluster stable siglet regions
with mlflow.start_run(run_name="cluster_analysis") as run:
    # Prepare data for clustering - we'll focus on regions with high truth scores
    # Create a DataFrame from our results
    data = []
    for i, theta in enumerate(thetas):
        for j, tau in enumerate(taus):
            data.append([theta, tau, truth_scores[i, j]])

    df = pd.DataFrame(data, columns=["theta", "tau", "truth_score"])

    # Log parameters
    stable_threshold = df["truth_score"].mean()  # Adjust threshold as needed
    mlflow.log_param("stable_threshold", stable_threshold)

    # Filter for stable regions (high truth scores)
    stable_regions = df[df["truth_score"] > stable_threshold]
    mlflow.log_metric("stable_regions_count", len(stable_regions))
    mlflow.log_metric("stable_regions_percentage", 100 * len(stable_regions) / len(df))

    # Normalize the data for clustering
    X = stable_regions[["theta", "tau"]].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Perform K-means clustering
    n_clusters = 5  # Adjust number of clusters as appropriate
    mlflow.log_param("n_clusters", n_clusters)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    stable_regions["cluster"] = kmeans.fit_predict(X_scaled)

    # Log metrics about the clusters
    for i in range(n_clusters):
        cluster_size = np.sum(stable_regions["cluster"] == i)
        mlflow.log_metric(f"cluster_{i}_size", cluster_size)
        mlflow.log_metric(
            f"cluster_{i}_mean_score",
            stable_regions[stable_regions["cluster"] == i]["truth_score"].mean(),
        )

    # Log inertia (sum of squared distances to centroids)
    mlflow.log_metric("kmeans_inertia", kmeans.inertia_)

    # Visualize the clusters
    plt.figure(figsize=(10, 8))
    for cluster in range(n_clusters):
        cluster_points = stable_regions[stable_regions["cluster"] == cluster]
        plt.scatter(
            cluster_points["tau"], cluster_points["theta"], label=f"Cluster {cluster}"
        )

    plt.xlabel("Coherence (τ)")
    plt.ylabel("Resonance (θ)")
    plt.title("Clusters of Stable Siglet Regions")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # Save figure locally
    clusters_fig_path = os.path.join(FIGURES_DIR, "siglet_clusters.png")
    plt.savefig(clusters_fig_path, dpi=300)

    # Log artifact
    mlflow.log_artifact(clusters_fig_path)

    plt.show()

In [ ]:
# Export results to CSV and save plots
with mlflow.start_run(run_name="export_results") as run:
    # Save the DataFrame with all results
    results_path = os.path.join(DATA_DIR, "siglet_qubit_results.csv")
    df.to_csv(results_path, index=False)
    mlflow.log_artifact(results_path)
    print(f"Exported results to {results_path}")

    # Save the clusters data
    clusters_path = os.path.join(DATA_DIR, "siglet_stable_clusters.csv")
    stable_regions.to_csv(clusters_path, index=False)
    mlflow.log_artifact(clusters_path)
    print(f"Exported clusters to {clusters_path}")

    # Log summary metrics
    mlflow.log_metric("total_datapoints", len(df))
    mlflow.log_metric("clustered_datapoints", len(stable_regions))
    mlflow.log_metric("clustering_coverage_pct", 100 * len(stable_regions) / len(df))

    # Create a summary of cluster centers
    cluster_centers = pd.DataFrame(
        kmeans.cluster_centers_, columns=["theta_scaled", "tau_scaled"]
    )

    # Inverse transform to get original theta and tau values
    centers_orig = scaler.inverse_transform(kmeans.cluster_centers_)
    cluster_centers["theta"] = centers_orig[:, 0]
    cluster_centers["tau"] = centers_orig[:, 1]

    # Add average truth score for each cluster
    cluster_centers["avg_truth_score"] = [
        stable_regions[stable_regions["cluster"] == i]["truth_score"].mean()
        for i in range(n_clusters)
    ]

    # Save cluster centers
    centers_path = os.path.join(DATA_DIR, "siglet_cluster_centers.csv")
    cluster_centers.to_csv(centers_path, index=False)
    mlflow.log_artifact(centers_path)
    print(f"Exported cluster centers to {centers_path}")

    # Display cluster centers
    print("\nSiglet Alphabet Candidates (Cluster Centers):")
    display(
        cluster_centers[["theta", "tau", "avg_truth_score"]].sort_values(
            "avg_truth_score", ascending=False
        )
    )

In [ ]:
# Generate complete decay curves for all parameter combinations
with mlflow.start_run(run_name="generate_decay_curves") as run:
    start_time = time.time()

    # Parameters for tracking
    mlflow.log_param("thetas_count", len(thetas))
    mlflow.log_param("taus_count", len(taus))
    mlflow.log_param("t_points", t_max + 1)

    # Create a matrix to store all decay curves
    # Shape: (n_curves, t_points) where n_curves = len(thetas) * len(taus)
    decay_curves = np.zeros((len(thetas) * len(taus), t_max + 1))

    # Parameters for each curve
    curve_params = []

    # Generate all decay curves
    curve_idx = 0
    for i, theta in enumerate(thetas):
        for j, tau in enumerate(taus):
            sq = SigletQubit(theta=theta, tau=tau, eps_norm=1.0, mu=0.8, c=0.7)
            decay_curves[curve_idx] = [sq.truth_score(t) for t in range(t_max + 1)]
            curve_params.append((theta, tau))
            curve_idx += 1

    # Convert parameters to numpy array for easier indexing
    curve_params = np.array(curve_params)

    # Log performance
    computation_time = time.time() - start_time
    mlflow.log_metric("decay_curves_generation_time", computation_time)

    # Save data for later use
    decay_curves_path = os.path.join(DATA_DIR, "decay_curves.npy")
    curve_params_path = os.path.join(DATA_DIR, "curve_params.npy")
    np.save(decay_curves_path, decay_curves)
    np.save(curve_params_path, curve_params)
    mlflow.log_artifact(decay_curves_path)
    mlflow.log_artifact(curve_params_path)

    # Plot a sample of curves to verify
    plt.figure(figsize=(12, 8))
    sample_indices = np.random.choice(len(decay_curves), 10, replace=False)
    for idx in sample_indices:
        theta, tau = curve_params[idx]
        plt.plot(
            range(t_max + 1), decay_curves[idx], label=f"θ={theta:.2f}, τ={tau:.2f}"
        )

    plt.xlabel("Time (t)")
    plt.ylabel("Truth Score T(s, t)")
    plt.title("Sample of Decay Curves")
    plt.legend()
    plt.grid(True)

    # Save and log the sample plot
    sample_fig_path = os.path.join(FIGURES_DIR, "sample_decay_curves.png")
    plt.savefig(sample_fig_path, dpi=300)
    mlflow.log_artifact(sample_fig_path)

    plt.show()

In [ ]:
# Compute DTW distances and apply clustering
with mlflow.start_run(run_name="dtw_clustering") as run:
    start_time = time.time()

    # Compute DTW distance matrix (this can be time-consuming)
    print("Computing DTW distance matrix...")
    dtw_dist = cdist_dtw(decay_curves)

    # Log computation time
    dtw_time = time.time() - start_time
    mlflow.log_metric("dtw_computation_time", dtw_time)
    print(f"DTW computation completed in {dtw_time:.2f} seconds")

    # Save DTW distance matrix
    np.save(os.path.join(DATA_DIR, "dtw_distances.npy"), dtw_dist)
    mlflow.log_artifact(os.path.join(DATA_DIR, "dtw_distances.npy"))

    # Normalize the DTW distances - add this line
    dtw_dist_normalized = dtw_dist / np.max(dtw_dist)

    # Try a range of eps values to find optimal clustering
    eps_values = [0.05, 0.1, 0.15, 0.2, 0.25]
    best_eps = None
    best_n_clusters = 0

    plt.figure(figsize=(15, 10))

    for i, eps in enumerate(eps_values):
        # Apply DBSCAN with different eps values on normalized distances
        dbscan = DBSCAN(metric="precomputed", eps=eps, min_samples=5)
        labels = dbscan.fit_predict(dtw_dist_normalized)  # Use normalized distances

        # Count number of clusters (excluding noise)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)

        # Log metrics
        mlflow.log_metric(f"dbscan_clusters_eps_{eps}", n_clusters)
        mlflow.log_metric(f"dbscan_noise_eps_{eps}", n_noise)

        # Plot in a subplot
        plt.subplot(1, len(eps_values), i + 1)

        # Plot cluster assignments in parameter space
        unique_labels = set(labels)
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

        for k, col in zip(unique_labels, colors):
            if k == -1:
                # Black used for noise
                col = "black"

            class_member_mask = labels == k
            xy = curve_params[class_member_mask]
            plt.scatter(
                xy[:, 1],
                xy[:, 0],
                s=10,
                c=[col],
                label=f"Cluster {k}" if k != -1 else "Noise",
            )

        plt.title(f"eps={eps}: {n_clusters} clusters, {n_noise} noise")
        plt.xlabel("Coherence (τ)")
        plt.ylabel("Resonance (θ)" if i == 0 else "")
        plt.grid(True)

        # Keep track of best eps value (maximum number of clusters, but not too many noise points)
        noise_percentage = n_noise / len(labels) * 100
        if n_clusters > best_n_clusters and noise_percentage < 50:
            best_n_clusters = n_clusters
            best_eps = eps

    plt.tight_layout()

    # Save the parameter exploration figure
    eps_exploration_path = os.path.join(FIGURES_DIR, "dbscan_eps_exploration.png")
    plt.savefig(eps_exploration_path, dpi=300)
    mlflow.log_artifact(eps_exploration_path)

    plt.show()

    # Use the best eps value for final clustering
    print(f"Best eps value: {best_eps} with {best_n_clusters} clusters")

    # Rerun DBSCAN with the best eps on normalized distances
    dbscan = DBSCAN(metric="precomputed", eps=best_eps, min_samples=5)
    labels_dbscan = dbscan.fit_predict(dtw_dist_normalized)  # Use normalized distances

    # Log final results
    n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    n_noise_dbscan = list(labels_dbscan).count(-1)
    mlflow.log_param("best_eps", best_eps)
    mlflow.log_metric("dbscan_n_clusters", n_clusters_dbscan)
    mlflow.log_metric("dbscan_n_noise_points", n_noise_dbscan)

    # Save final labels
    labels_path = os.path.join(DATA_DIR, "dbscan_labels.npy")
    np.save(labels_path, labels_dbscan)
    mlflow.log_artifact(labels_path)

In [ ]:
# Let's verify this shape invariance hypothesis
with mlflow.start_run(run_name="shape_analysis") as run:
    # Normalize each curve to [0,1] range to compare pure shapes
    normalized_curves = np.zeros_like(decay_curves)
    for i in range(len(decay_curves)):
        curve = decay_curves[i]
        min_val = np.min(curve)
        max_val = np.max(curve)
        if max_val > min_val:  # Avoid division by zero
            normalized_curves[i] = (curve - min_val) / (max_val - min_val)
        else:
            normalized_curves[i] = curve

    # Compute DTW distances on normalized curves
    print("Computing DTW distances on normalized curves...")
    norm_dtw_dist = cdist_dtw(normalized_curves)

    # Visualize the distribution of distances
    plt.figure(figsize=(10, 8))
    plt.hist(norm_dtw_dist.flatten(), bins=50)
    plt.xlabel("DTW Distance (normalized curves)")
    plt.ylabel("Frequency")
    plt.title("Distribution of Shape Differences Between Curves")
    plt.grid(True)

    # Save figure
    shape_dist_path = os.path.join(FIGURES_DIR, "shape_distance_distribution.png")
    plt.savefig(shape_dist_path, dpi=300)
    mlflow.log_artifact(shape_dist_path)
    plt.show()

    # Plot a sample of normalized curves
    plt.figure(figsize=(12, 8))
    sample_indices = np.random.choice(len(normalized_curves), 20, replace=False)
    for idx in sample_indices:
        theta, tau = curve_params[idx]
        plt.plot(
            range(t_max + 1),
            normalized_curves[idx],
            alpha=0.7,
            label=f"θ={theta:.2f}, τ={tau:.2f}" if idx == sample_indices[0] else "",
        )

    plt.xlabel("Time (t)")
    plt.ylabel("Normalized Truth Score")
    plt.title("Shape Comparison of Normalized Decay Curves")
    if len(sample_indices) > 0:
        plt.legend()
    plt.grid(True)

    # Save figure
    norm_curves_path = os.path.join(FIGURES_DIR, "normalized_curves_comparison.png")
    plt.savefig(norm_curves_path, dpi=300)
    mlflow.log_artifact(norm_curves_path)
    plt.show()

    # Calculate the mean curve and distance from mean
    mean_curve = np.mean(normalized_curves, axis=0)
    distances_from_mean = []

    for curve in normalized_curves:
        dist = np.sqrt(np.sum((curve - mean_curve) ** 2))
        distances_from_mean.append(dist)

    # Plot mean curve and distribution of distances
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.plot(range(t_max + 1), mean_curve, "r-", linewidth=3)
    plt.xlabel("Time (t)")
    plt.ylabel("Normalized Truth Score")
    plt.title("Mean Curve Shape")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.hist(distances_from_mean, bins=30)
    plt.xlabel("Euclidean Distance from Mean Curve")
    plt.ylabel("Frequency")
    plt.title("Curve Variation Around Mean Shape")
    plt.grid(True)

    plt.tight_layout()

    # Save figure
    mean_curve_path = os.path.join(FIGURES_DIR, "mean_curve_analysis.png")
    plt.savefig(mean_curve_path, dpi=300)
    mlflow.log_artifact(mean_curve_path)
    plt.show()

    # Log the scientific finding
    mlflow.log_metric("mean_shape_distance", np.mean(distances_from_mean))
    mlflow.log_metric("max_shape_distance", np.max(distances_from_mean))

    # Scientific conclusion
    conclusion = """
    # Shape Invariance in Siglet-Qubit System
    
    ## Finding
    
    The DTW-based clustering reveals that despite varying parameters (θ,τ), all decay curves 
    belong to the same shape family. This suggests a fundamental shape invariant in the 
    siglet-qubit temporal dynamics.
    
    ## Mathematical Explanation
    
    The truth score function T(s,t) = cos(theta) * tau^t * eps_norm * mu * c creates curves 
    that share the same underlying exponential decay pattern, with differences only in:
    
    1. Initial amplitude (controlled by θ)
    2. Decay rate (controlled by τ)
    
    ## Implications
    
    This shape invariance suggests that siglet-qubits with different parameters maintain 
    temporal coherence in their dynamic behavior, a potentially important property for 
    quantum information processing.
    
    The siglet alphabet can be defined by (θ,τ) parameter clusters, with the understanding 
    that temporal dynamics follow a universal pattern across all symbols.
    """

    with open(os.path.join(DATA_DIR, "shape_invariance_finding.md"), "w") as f:
        f.write(conclusion)

    mlflow.log_artifact(os.path.join(DATA_DIR, "shape_invariance_finding.md"))

In [ ]:
# Visualize DBSCAN clusters
with mlflow.start_run(run_name="dbscan_visualization") as run:
    # Identify unique clusters (excluding noise points)
    unique_clusters = sorted(set(labels_dbscan))
    if -1 in unique_clusters:
        unique_clusters.remove(-1)

    # Create a colormap for clusters
    colors = cm.viridis(np.linspace(0, 1, len(unique_clusters)))

    # 1. Plot representative curves from each cluster
    plt.figure(figsize=(15, 10))

    # First plot noise points (if any)
    noise_indices = np.where(labels_dbscan == -1)[0]
    if len(noise_indices) > 0:
        # Sample at most 5 noise points
        sample_noise = np.random.choice(
            noise_indices, min(5, len(noise_indices)), replace=False
        )
        for idx in sample_noise:
            plt.plot(range(t_max + 1), decay_curves[idx], "k--", alpha=0.3)

    # Plot curves from each cluster
    for i, cluster in enumerate(unique_clusters):
        # Get indices for this cluster
        cluster_indices = np.where(labels_dbscan == cluster)[0]

        # Choose representative curves (cluster center + random samples)
        # Find curve closest to cluster center by minimizing sum of DTW distances
        if len(cluster_indices) > 1:
            within_cluster_dist = dtw_dist[np.ix_(cluster_indices, cluster_indices)]
            center_idx = cluster_indices[np.argmin(within_cluster_dist.sum(axis=1))]

            # Plot cluster center
            plt.plot(
                range(t_max + 1),
                decay_curves[center_idx],
                color=colors[i],
                linewidth=3,
                label=f"Cluster {cluster} (center)",
            )

            # Plot a few random members
            sample_size = min(3, len(cluster_indices) - 1)
            if sample_size > 0:
                other_indices = [idx for idx in cluster_indices if idx != center_idx]
                sample_indices = np.random.choice(
                    other_indices, sample_size, replace=False
                )
                for idx in sample_indices:
                    plt.plot(
                        range(t_max + 1), decay_curves[idx], color=colors[i], alpha=0.5
                    )
        else:
            # If only one curve in cluster
            plt.plot(
                range(t_max + 1),
                decay_curves[cluster_indices[0]],
                color=colors[i],
                linewidth=3,
                label=f"Cluster {cluster}",
            )

    plt.xlabel("Time (t)")
    plt.ylabel("Truth Score T(s, t)")
    plt.title("Representative Decay Curves by DBSCAN Cluster")
    plt.legend()
    plt.grid(True)

    # Save figure
    dbscan_curves_fig_path = os.path.join(
        FIGURES_DIR, "dbscan_representative_curves.png"
    )
    plt.savefig(dbscan_curves_fig_path, dpi=300)
    mlflow.log_artifact(dbscan_curves_fig_path)

    plt.show()

    # 2. Plot clusters in parameter space (θ,τ)
    plt.figure(figsize=(12, 10))

    # Plot noise points first
    if len(noise_indices) > 0:
        noise_params = curve_params[noise_indices]
        plt.scatter(
            noise_params[:, 1],
            noise_params[:, 0],
            c="gray",
            marker="x",
            alpha=0.5,
            label="Noise",
        )

    # Plot each cluster
    for i, cluster in enumerate(unique_clusters):
        cluster_indices = np.where(labels_dbscan == cluster)[0]
        cluster_params = curve_params[cluster_indices]
        plt.scatter(
            cluster_params[:, 1],
            cluster_params[:, 0],
            c=[colors[i]],
            label=f"Cluster {cluster}",
        )

    plt.xlabel("Coherence (τ)")
    plt.ylabel("Resonance (θ)")
    plt.title("DBSCAN Clusters in Parameter Space")
    plt.legend()
    plt.grid(True)

    # Save figure
    dbscan_params_fig_path = os.path.join(FIGURES_DIR, "dbscan_parameter_space.png")
    plt.savefig(dbscan_params_fig_path, dpi=300)
    mlflow.log_artifact(dbscan_params_fig_path)

    plt.show()

In [ ]:
# Visualize Spectral Clustering results
with mlflow.start_run(run_name="spectral_visualization") as run:
    # Create a colormap for clusters
    colors = cm.plasma(np.linspace(0, 1, n_clusters_spectral))

    # 1. Plot representative curves from each cluster
    plt.figure(figsize=(15, 10))

    # Plot curves from each cluster
    for cluster in range(n_clusters_spectral):
        # Get indices for this cluster
        cluster_indices = np.where(labels_spectral == cluster)[0]

        # Find curve closest to cluster center
        within_cluster_dist = dtw_dist[np.ix_(cluster_indices, cluster_indices)]
        center_idx = cluster_indices[np.argmin(within_cluster_dist.sum(axis=1))]

        # Plot cluster center
        plt.plot(
            range(t_max + 1),
            decay_curves[center_idx],
            color=colors[cluster],
            linewidth=3,
            label=f"Cluster {cluster} (center)",
        )

        # Plot a few random members
        sample_size = min(3, len(cluster_indices) - 1)
        if sample_size > 0:
            other_indices = [idx for idx in cluster_indices if idx != center_idx]
            sample_indices = np.random.choice(other_indices, sample_size, replace=False)
            for idx in sample_indices:
                plt.plot(
                    range(t_max + 1),
                    decay_curves[idx],
                    color=colors[cluster],
                    alpha=0.5,
                )

    plt.xlabel("Time (t)")
    plt.ylabel("Truth Score T(s, t)")
    plt.title("Representative Decay Curves by Spectral Cluster")
    plt.legend()
    plt.grid(True)

    # Save figure
    spectral_curves_fig_path = os.path.join(
        FIGURES_DIR, "spectral_representative_curves.png"
    )
    plt.savefig(spectral_curves_fig_path, dpi=300)
    mlflow.log_artifact(spectral_curves_fig_path)

    plt.show()

    # 2. Plot clusters in parameter space (θ,τ)
    plt.figure(figsize=(12, 10))

    # Plot each cluster
    for cluster in range(n_clusters_spectral):
        cluster_indices = np.where(labels_spectral == cluster)[0]
        cluster_params = curve_params[cluster_indices]
        plt.scatter(
            cluster_params[:, 1],
            cluster_params[:, 0],
            c=[colors[cluster]],
            label=f"Cluster {cluster}",
        )

    plt.xlabel("Coherence (τ)")
    plt.ylabel("Resonance (θ)")
    plt.title("Spectral Clusters in Parameter Space")
    plt.legend()
    plt.grid(True)

    # Save figure
    spectral_params_fig_path = os.path.join(FIGURES_DIR, "spectral_parameter_space.png")
    plt.savefig(spectral_params_fig_path, dpi=300)
    mlflow.log_artifact(spectral_params_fig_path)

    plt.show()

In [ ]:
# 3D visualization combining time series shape and parameter space
with mlflow.start_run(run_name="3d_visualization") as run:
    # Create 3D plot for DBSCAN
    fig = plt.figure(figsize=(15, 12))
    ax = fig.add_subplot(111, projection="3d")

    # Add cluster information to parameter data
    param_cluster_data = np.column_stack((curve_params, labels_dbscan))

    # Exclude noise points for clearer visualization
    valid_data = param_cluster_data[param_cluster_data[:, 2] != -1]

    # Extract parameters and cluster labels
    thetas_valid = valid_data[:, 0]
    taus_valid = valid_data[:, 1]
    labels_valid = valid_data[:, 2].astype(int)

    # Compute a representative value for each curve (e.g., decay rate)
    decay_rates = []
    for idx in range(len(decay_curves)):
        if labels_dbscan[idx] != -1:  # Skip noise points
            curve = decay_curves[idx]
            # Calculate decay rate (change over time)
            decay_rate = (curve[0] - curve[-1]) / t_max
            decay_rates.append(decay_rate)

    # Create scatter plot
    unique_clusters = sorted(set(labels_valid))
    colors = cm.viridis(np.linspace(0, 1, len(unique_clusters)))

    for i, cluster in enumerate(unique_clusters):
        cluster_mask = labels_valid == cluster
        ax.scatter(
            taus_valid[cluster_mask],
            thetas_valid[cluster_mask],
            decay_rates[np.where(cluster_mask)[0]],
            c=[colors[i]],
            label=f"Cluster {int(cluster)}",
        )

    ax.set_xlabel("Coherence (τ)")
    ax.set_ylabel("Resonance (θ)")
    ax.set_zlabel("Decay Rate")
    ax.set_title("DBSCAN Clusters: Parameter Space + Decay Dynamics")
    plt.legend()

    # Save figure
    dbscan_3d_fig_path = os.path.join(FIGURES_DIR, "dbscan_3d_visualization.png")
    plt.savefig(dbscan_3d_fig_path, dpi=300)
    mlflow.log_artifact(dbscan_3d_fig_path)

    plt.show()

    # Create comparison table of clustering results
    cluster_stats = {
        "method": ["DBSCAN", "Spectral"],
        "n_clusters": [n_clusters_dbscan, n_clusters_spectral],
        "noise_points": [n_noise_dbscan, 0],  # Spectral assigns all points to clusters
    }

    # Create DataFrame
    stats_df = pd.DataFrame(cluster_stats)

    # Display and save
    print("Clustering Method Comparison:")
    display(stats_df)

    # Save to CSV
    stats_path = os.path.join(DATA_DIR, "clustering_comparison.csv")
    stats_df.to_csv(stats_path, index=False)
    mlflow.log_artifact(stats_path)

    # Log summary metrics
    mlflow.log_metric("dbscan_decay_shape_clusters", n_clusters_dbscan)
    mlflow.log_metric("spectral_decay_shape_clusters", n_clusters_spectral)

In [ ]:
# Compare parameter-based clustering with shape-based clustering
with mlflow.start_run(run_name="cluster_comparison") as run:
    # Create a comparison visualization
    plt.figure(figsize=(15, 10))

    # Setup subplots
    ax1 = plt.subplot(221)  # Parameter K-means
    ax2 = plt.subplot(222)  # DTW DBSCAN
    ax3 = plt.subplot(223)  # DTW Spectral
    ax4 = plt.subplot(224)  # Agreement heatmap

    # 1. Plot original parameter-based K-means clusters
    for cluster in range(n_clusters):  # From your original K-means
        cluster_points = stable_regions[stable_regions["cluster"] == cluster]
        ax1.scatter(
            cluster_points["tau"], cluster_points["theta"], label=f"Cluster {cluster}"
        )

    ax1.set_xlabel("Coherence (τ)")
    ax1.set_ylabel("Resonance (θ)")
    ax1.set_title("Parameter Space K-means")
    ax1.legend()
    ax1.grid(True)

    # 2. Plot DTW DBSCAN clusters
    # Get valid points (excluding noise)
    valid_indices = np.where(labels_dbscan != -1)[0]
    valid_params = curve_params[valid_indices]
    valid_labels = labels_dbscan[valid_indices]

    # Create colormap for DBSCAN
    dbscan_colors = cm.viridis(np.linspace(0, 1, len(set(valid_labels))))

    for i, cluster in enumerate(sorted(set(valid_labels))):
        cluster_mask = valid_labels == cluster
        ax2.scatter(
            valid_params[cluster_mask, 1],
            valid_params[cluster_mask, 0],
            c=[dbscan_colors[i]],
            label=f"Cluster {int(cluster)}",
        )

    ax2.set_xlabel("Coherence (τ)")
    ax2.set_ylabel("Resonance (θ)")
    ax2.set_title("DTW DBSCAN Clusters")
    ax2.legend()
    ax2.grid(True)

    # 3. Plot DTW Spectral clusters
    spectral_colors = cm.plasma(np.linspace(0, 1, n_clusters_spectral))

    for cluster in range(n_clusters_spectral):
        cluster_indices = np.where(labels_spectral == cluster)[0]
        cluster_params = curve_params[cluster_indices]
        ax3.scatter(
            cluster_params[:, 1],
            cluster_params[:, 0],
            c=[spectral_colors[cluster]],
            label=f"Cluster {cluster}",
        )

    ax3.set_xlabel("Coherence (τ)")
    ax3.set_ylabel("Resonance (θ)")
    ax3.set_title("DTW Spectral Clusters")
    ax3.legend()
    ax3.grid(True)

    # 4. Create confusion/agreement matrix between DBSCAN and Spectral
    # This shows how often they agree on cluster assignments

    # Filter for points that aren't noise in DBSCAN
    non_noise_idx = np.where(labels_dbscan != -1)[0]
    dbscan_filtered = labels_dbscan[non_noise_idx]
    spectral_filtered = labels_spectral[non_noise_idx]

    # Create contingency table
    contingency = confusion_matrix(dbscan_filtered, spectral_filtered)

    # Normalize by rows (DBSCAN clusters)
    contingency_norm = (
        contingency.astype("float") / contingency.sum(axis=1)[:, np.newaxis]
    )

    # Plot heatmap
    im = ax4.imshow(contingency_norm, cmap="Blues")

    # Add labels
    ax4.set_xlabel("Spectral Clusters")
    ax4.set_ylabel("DBSCAN Clusters")
    ax4.set_title("Cluster Assignment Agreement")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax4)
    cbar.set_label("Proportion")

    # Adjust layout
    plt.tight_layout()

    # Save figure
    comparison_fig_path = os.path.join(FIGURES_DIR, "clustering_method_comparison.png")
    plt.savefig(comparison_fig_path, dpi=300)
    mlflow.log_artifact(comparison_fig_path)

    plt.show()

    # Compute agreement metrics
    ari = adjusted_rand_score(dbscan_filtered, spectral_filtered)
    ami = adjusted_mutual_info_score(dbscan_filtered, spectral_filtered)

    mlflow.log_metric("adjusted_rand_index", ari)
    mlflow.log_metric("adjusted_mutual_info", ami)

    print(f"Adjusted Rand Index: {ari:.4f}")
    print(f"Adjusted Mutual Information: {ami:.4f}")

    # Create summary of findings
    summary = f"""
    # Siglet Shape Clustering Summary
    
    ## Key Findings
    
    1. **DBSCAN identified {n_clusters_dbscan} natural clusters** based on decay curve shapes
       - {n_noise_dbscan} points classified as noise
       
    2. **Spectral Clustering produced {n_clusters_spectral} clusters** with smoother boundaries
    
    3. **Agreement between methods: ARI={ari:.4f}, AMI={ami:.4f}**
       - Higher values indicate more agreement (max=1.0)
    
    4. **Shape vs. Parameter Clustering**
       - Parameter clustering: Based on (θ,τ) coordinates
       - Shape clustering: Based on temporal dynamics
       
    ## Implications for Siglet Alphabet
    
    The stable regions in parameter space correspond to distinct temporal behaviors.
    These regions form the foundation for the siglet alphabet where each symbol has:
    
    1. A characteristic resonance angle (θ)
    2. A coherence factor (τ)
    3. A distinct temporal decay pattern
    
    ## Next Steps
    
    1. Fine-tune DTW parameters for optimal shape separation
    2. Export selected curves as canonical siglet prototypes
    3. Map to quantum circuit parameters
    """

    summary_path = os.path.join(NOTES_DIR, "siglet_shape_clustering_summary.md")
    with open(summary_path, "w") as f:
        f.write(summary)

    mlflow.log_artifact("siglet_shape_clustering_summary.md")

In [ ]:
# Next steps for quantum extension (preparation for tomorrow's work)
with mlflow.start_run(run_name="quantum_extension_prep") as run:
    # Import quantum libraries (commented out as they're for future use)
    # import qiskit
    # from qiskit import QuantumCircuit, Aer, execute
    # import pennylane as qml

    # Define a basic quantum circuit version of the SigletQubit
    def quantum_siglet_circuit(theta, tau):
        """
        Create a quantum circuit that implements the SigletQubit

        Args:
            theta (float): Resonance angle
            tau (float): Coherence factor

        Returns:
            QuantumCircuit: A Qiskit circuit implementing the siglet
        """
        # This is a placeholder for future quantum implementation
        # Example basic circuit structure:
        """
        qc = QuantumCircuit(1, 1)
        qc.rx(theta, 0)  # Apply resonance angle
        # Apply decoherence modeling based on tau
        # ...
        qc.measure(0, 0)
        return qc
        """
        pass

    # Log notes for tomorrow's work
    notes = """
    Tomorrow's quantum extension will:
    1. Port the SigletQubit model to a 1-qubit quantum circuit
    2. Map the classical parameters (theta, tau) to quantum gate parameters
    3. Test the quantum version against our classical simulation
    4. Evaluate decoherence effects in actual quantum systems vs our model
    """

    notes_path = os.path.join(NOTES_DIR, "quantum_extension_notes.md")
    with open(notes_path, "w") as f:
        f.write(notes)

    mlflow.log_artifact("quantum_extension_notes.md")

    # Log the cluster centers as potential quantum parameter targets
    mlflow.log_artifact(centers_path, "quantum_parameter_targets")

    print("Preparation for quantum extension complete.")
    print(
        "Identified stable siglet parameters as candidates for quantum implementation."
    )